# Scenario  

You are a Data Engineer supporting an enterprise Technology Operations Center(TOC).  

Every morning, management want an automated report that summarizes the previous week's operational performance.  

Instead of manually opening Excel and creating reports, your job is to build a Python reporting pipeline that:
- Cleans operational data
- Calculates business metrics
- Builds multiple reports
- Exports reports to CSV and Excel
- Produces a summary text report

In [3]:
import os
import pandas as pd
from datetime import datetime

# Create a folder for exported reports
reports_folder = "Week_5_Mini_Project"

os.makedirs(
    reports_folder,
    exist_ok=True
)

incident_data = {
    "incident_id": [
        "INC001","INC002","INC003","INC004","INC005",
        "INC006","INC007","INC008","INC009","INC010",
        "INC011","INC012","INC013","INC014","INC015"
    ],

    "server": [
        "WEB01","DB01","APP01","WEB02","DB02",
        "APP02","WEB03","WEB01","APP03","DB01",
        "WEB04","APP01","DB03","WEB02","APP02"
    ],

    "team": [
        "Web","Database","Application","Web","Database",
        "Application","Web","Web","Application","Database",
        "Web","Application","Database","Web","Application"
    ],

    "environment": [
        "Production","Development","Production","Production","Development",
        "Development","Production","Production","Development","Development",
        "Production","Production","Development","Production","Development"
    ],

    "priority": [
        "Critical","High","Medium","High","Critical",
        "Low","Medium","Critical","High","Medium",
        "Low","Critical","High","Medium","Low"
    ],

    "status": [
        "Down","Warning","Down","Warning","Down",
        "Up","Warning","Down","Up","Warning",
        "Down","Down","Warning","Up","Down"
    ],

    "cpu_usage": [
        96,82,91,74,94,
        48,79,97,55,72,
        89,93,87,60,92
    ],

    "memory_usage": [
        91,73,89,70,90,
        46,76,95,58,68,
        84,90,82,61,88
    ],

    "incident_time": [
        "2026-07-01 08:15:00",
        "2026-07-01 10:20:00",
        "2026-07-01 14:30:00",
        "2026-07-02 09:10:00",
        "2026-07-02 13:15:00",
        "2026-07-02 17:05:00",
        "2026-07-03 08:00:00",
        "2026-07-03 11:40:00",
        "2026-07-04 09:30:00",
        "2026-07-04 15:10:00",
        "2026-07-05 08:25:00",
        "2026-07-05 12:45:00",
        "2026-07-06 09:50:00",
        "2026-07-06 14:05:00",
        "2026-07-07 10:15:00"
    ],

    "resolved_time": [
        "2026-07-01 09:45:00",
        "2026-07-01 10:50:00",
        "2026-07-01 15:15:00",
        "2026-07-02 09:45:00",
        "2026-07-02 14:35:00",
        "2026-07-02 17:20:00",
        "2026-07-03 08:40:00",
        "2026-07-03 13:00:00",
        "2026-07-04 09:55:00",
        "2026-07-04 15:50:00",
        "2026-07-05 09:05:00",
        "2026-07-05 14:20:00",
        "2026-07-06 10:30:00",
        "2026-07-06 14:40:00",
        "2026-07-07 11:50:00"
    ]
}

df = pd.DataFrame(incident_data)

print(df)

   incident_id server         team  environment  priority   status  cpu_usage  \
0       INC001  WEB01          Web   Production  Critical     Down         96   
1       INC002   DB01     Database  Development      High  Warning         82   
2       INC003  APP01  Application   Production    Medium     Down         91   
3       INC004  WEB02          Web   Production      High  Warning         74   
4       INC005   DB02     Database  Development  Critical     Down         94   
5       INC006  APP02  Application  Development       Low       Up         48   
6       INC007  WEB03          Web   Production    Medium  Warning         79   
7       INC008  WEB01          Web   Production  Critical     Down         97   
8       INC009  APP03  Application  Development      High       Up         55   
9       INC010   DB01     Database  Development    Medium  Warning         72   
10      INC011  WEB04          Web   Production       Low     Down         89   
11      INC012  APP01  Appli

In [6]:
# Prepare the Data - Convert incident_time & resolved_time to datetime
# Create incident_date, resolution_minutes, and incident_weekday

df["incident_time"] = pd.to_datetime(
    df["incident_time"]
)

df["resolved_time"] = pd.to_datetime(
    df["resolved_time"]
)

df["incident_date"] = (
    df["incident_time"].dt.date
)

df["incident_weekday"] = (
    df["incident_time"].dt.day_name()
)

df["resolution_minutes"] = (
    (
        df["resolved_time"]
        - df["incident_time"]
    )
    .dt.total_seconds()
    / 60
)

print(
    df[
        [
            "incident_id",
            "server",
            "team",
            "environment",
            "status",
            "priority",
            "incident_date",
            "incident_weekday",
            "resolution_minutes"
        ]
    ]
)



   incident_id server         team  environment   status  priority  \
0       INC001  WEB01          Web   Production     Down  Critical   
1       INC002   DB01     Database  Development  Warning      High   
2       INC003  APP01  Application   Production     Down    Medium   
3       INC004  WEB02          Web   Production  Warning      High   
4       INC005   DB02     Database  Development     Down  Critical   
5       INC006  APP02  Application  Development       Up       Low   
6       INC007  WEB03          Web   Production  Warning    Medium   
7       INC008  WEB01          Web   Production     Down  Critical   
8       INC009  APP03  Application  Development       Up      High   
9       INC010   DB01     Database  Development  Warning    Medium   
10      INC011  WEB04          Web   Production     Down       Low   
11      INC012  APP01  Application   Production     Down  Critical   
12      INC013   DB03     Database  Development  Warning      High   
13      INC014  WEB0

In [7]:
# Create Business Metrics - Create a Boolean Column sla_breached where resolution minutes > 60
# Create another Boolean column high_cpu where cpu_usage >= 90 to simulate business KPIs

df["sla_breached"] = (
    df["resolution_minutes"] > 60
)

df["high_cpu"] = (
    df["cpu_usage"] >= 90
)

print(
    df[
        [
            "incident_id",
            "resolution_minutes",
            "sla_breached",
            "cpu_usage",
            "high_cpu"
        ]
    ]
)

   incident_id  resolution_minutes  sla_breached  cpu_usage  high_cpu
0       INC001                90.0          True         96      True
1       INC002                30.0         False         82     False
2       INC003                45.0         False         91      True
3       INC004                35.0         False         74     False
4       INC005                80.0          True         94      True
5       INC006                15.0         False         48     False
6       INC007                40.0         False         79     False
7       INC008                80.0          True         97      True
8       INC009                25.0         False         55     False
9       INC010                40.0         False         72     False
10      INC011                40.0         False         89     False
11      INC012                95.0          True         93      True
12      INC013                40.0         False         87     False
13      INC014      

In [9]:
# Report 1 - Create team_report to include: incident_count, average_cpu, highest_cpu, average_memory,
# average_resolution_minutes, sla_breaches; Requirements: round(2), reset_index, sort by incident_count desc

team_report = (
    df.groupby("team")
      .agg(
          incident_count=("incident_id", "count"),
          average_cpu=("cpu_usage", "mean"),
          highest_cpu=("cpu_usage", "max"),
          average_memory=("memory_usage", "mean"),
          sla_breaches=("sla_breached", "sum"),
          average_resolution_minutes=(
              "resolution_minutes",
              "mean"
          )
      )
      .reset_index()
      .round(2)
      .sort_values(
          by="incident_count",
          ascending=False
      )
)

print(team_report)

          team  incident_count  average_cpu  highest_cpu  average_memory  \
2          Web               6        82.50           97           79.50   
0  Application               5        75.80           93           74.20   
1     Database               4        83.75           94           78.25   

   sla_breaches  average_resolution_minutes  
2             2                       53.33  
0             2                       55.00  
1             1                       47.50  


In [10]:
# Report 2 - Create environment_report to include: incident_count, average_cpu, average_memory,
# high_cpu_servers, average_resolution_minutes. Use lamba for high_cpu_servers

environment_report = (
    df.groupby("environment")
      .agg(
          incident_count=("incident_id", "count"),
          average_cpu=("cpu_usage", "mean"),
          average_memory=("memory_usage", "mean"),
          high_cpu_servers=(
              "priority",
              lambda values: (
                  values == "Critical"
              ).sum()
          ),
          average_resolution_minutes=(
              "resolution_minutes",
              "mean"
          )
      )
      .reset_index()
      .round(2)
)

print(environment_report)

   environment  incident_count  average_cpu  average_memory  high_cpu_servers  \
0  Development               7        75.71           72.14                 1   
1   Production               8        84.88           82.00                 3   

   average_resolution_minutes  
0                       46.43  
1                       57.50  


In [11]:
# Report 3 - Daily Operations Report. Group by incident_date. Calculate incident_count, average_cpu,
# highest_cpu, average_resolution_minutes, sla_breaches

daily_ops_report = (
    df.groupby("incident_date")
      .agg(
          incident_count=("incident_id", "count"),
          average_cpu=("cpu_usage", "mean"),
          highest_cpu=("cpu_usage", "max"),
          sla_breaches=("sla_breached", "count"),
          average_resolution_minutes=(
              "resolution_minutes",
              "mean"
          )
      )
      .reset_index()
      .round(2)
)

print(daily_ops_report)

  incident_date  incident_count  average_cpu  highest_cpu  sla_breaches  \
0    2026-07-01               3        89.67           96             3   
1    2026-07-02               3        72.00           94             3   
2    2026-07-03               2        88.00           97             2   
3    2026-07-04               2        63.50           72             2   
4    2026-07-05               2        91.00           93             2   
5    2026-07-06               2        73.50           87             2   
6    2026-07-07               1        92.00           92             1   

   average_resolution_minutes  
0                       55.00  
1                       43.33  
2                       60.00  
3                       32.50  
4                       67.50  
5                       37.50  
6                       95.00  


In [12]:
# Report 4 - Priority Pivot Report. Create a pivot table where rows:priority; columns:status;
# values:incident_id; aggregation:count; fill missing values, include totals, rename totals to "Total"

priority_pivot_report = (
    pd.pivot_table(
        data=df,
        index="priority",
        columns="status",
        values="incident_id",
        aggfunc="count",
        fill_value=0,
        margins=True,
        margins_name="Total"
    )
    .reset_index()
)

priority_pivot_report.columns.name = None

print(priority_pivot_report)

   priority  Down  Up  Warning  Total
0  Critical     4   0        0      4
1      High     0   1        3      4
2       Low     2   1        0      3
3    Medium     1   1        2      4
4     Total     7   3        5     15


In [13]:
# Export Reports - Export every report as CSV into Reports.

team_report_path = os.path.join(
    reports_folder,
    "team_performance_report.csv"
)

environment_report_path = os.path.join(
    reports_folder,
    "environment_summary.csv"
)

daily_ops_report_path = os.path.join(
    reports_folder,
    "daily_ops_report.csv"
)

priority_pivot_report_path = os.path.join(
    reports_folder,
    "priority_pivot_report.csv"
)


team_report.to_csv(
    team_report_path,
    index=False
)

environment_report.to_csv(
    environment_report_path,
    index=False
)

daily_ops_report.to_csv(
    daily_ops_report_path,
    index=False
)

priority_pivot_report.to_csv(
    priority_pivot_report_path,
    index=False
)

print("CSV reports exported successfully.")

CSV reports exported successfully.


In [15]:
# Confirm Files Exist

print(os.path.exists(team_report_path))
print(os.path.exists(environment_report_path))
print(os.path.exists(daily_ops_report_path))
print(os.path.exists(priority_pivot_report_path))

True
True
True
True


In [16]:
# Excel Workbook - Create operations_dashboard.xlsx. Include worksheets: Team Report, Environment Report,
# Daily Report, Priority Pivot

excel_report_path = os.path.join(
    reports_folder,
    "operations_dashboard.xlsx"
)

with pd.ExcelWriter(
    excel_report_path,
    engine="openpyxl"
) as writer:

    team_report.to_excel(
        writer,
        sheet_name="Team Report",
        index=False
    )

    environment_report.to_excel(
    writer,
    sheet_name="Environment Report",
    index=False
    )

    daily_ops_report.to_excel(
    writer,
    sheet_name="Daily Report",
    index=False
    )

    priority_pivot_report.to_excel(
    writer,
    sheet_name="Priority Pivot",
    index=False
    )

print("Excel report saved:", excel_report_path)

Excel report saved: Week_5_Mini_Project/operations_dashboard.xlsx


In [17]:
# Text Executive Summary - Create operations_summary.txt. Include: data generated, total incidents, critical
# incidents, average CPU, average resolution time, number of SLA breaches, number of high CPU incidents

generated_time = datetime.now().strftime(
    "%Y-%m-%d %H:%M:%S"
)

total_incidents = len(df)

critical_incidents = (
    df["priority"] == "Critical"
).sum()

average_cpu = (
    df["cpu_usage"].mean()
)

average_resolution = (
    df["resolution_minutes"].mean()
)

total_sla_breaches = (
    df["sla_breached"].sum()
)

high_cpu_incidents = (
    df["cpu_usage"].max()
)

text_summary_path = os.path.join(
    reports_folder,
    "operations_summary.txt"
)

with open(
    text_summary_path,
    "w"
) as report_file:

    report_file.write(
        "OPERATIONS SUMMARY\n"
    )

    report_file.write(
        "=========================================\n\n"
    )

    report_file.write(
        f"Date Generated: {generated_time}\n\n"
    )

    report_file.write(
        f"Total Incidents: {total_incidents}\n"
    )

    report_file.write(
        f"Critical Incidents: {critical_incidents}\n"
    )

    report_file.write(
        f"Average CPU: {average_cpu}\n"
    )

    report_file.write(
        f"Average Resolution Time: "
        f"{average_resolution:.2f} minutes\n"
    )

    report_file.write(
        f"SLA Breaches: {total_sla_breaches}\n"
    )

    report_file.write(
        f"High CPU Incidents: {high_cpu_incidents}\n"
    )

print("Text summary saved:", text_summary_path)
    

Text summary saved: Week_5_Mini_Project/operations_summary.txt


In [19]:
# Validation - Verify that every exported CSV exists, Verify the Excel workbook exists, verify the text
# report exists, print every file inside the reports folder in alphabetical order

exported_files = sorted(
    os.listdir(reports_folder)
)

for file_name in exported_files:
    print(file_name)

.ipynb_checkpoints
daily_ops_report.csv
environment_summary.csv
operations_dashboard.xlsx
operations_summary.txt
priority_pivot_report.csv
team_performance_report.csv


In [20]:
# Add timestamp to filenames

report_timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

print(report_timestamp)

timestamped_excel_path = os.path.join(
    reports_folder,
    f"operations_dashboard_{report_timestamp}.xlsx"
)

timestamped_csv_path = os.path.join(
    reports_folder,
    f"daily_ops_report_{report_timestamp}.csv",
    f"environment_summary_{report_timestamp}.csv",
    f"operations_summary_{report_timestamp}.csv",
    f"priority_pivot_report_{report_timestamp}.csv",
    f"team_performance_report_{report_timestamp}.csv"
)

print(timestamped_excel_path)
print(timestamped_csv_path)

20260720_221648
Week_5_Mini_Project/operations_dashboard_20260720_221648.xlsx
Week_5_Mini_Project/daily_ops_report_20260720_221648.csv/environment_summary_20260720_221648.csv/operations_summary_20260720_221648.csv/priority_pivot_report_20260720_221648.csv/team_performance_report_20260720_221648.csv
